# Lecture 11 Activity – Which street has the most fatal crashes?

QSS 20 · Modern Statistical Computing · Fall 2026

## The question

In lecture 10's Activity 1, `GROUP BY street` ranked the streets by their
fatal crashes. It counted spellings, not streets: `NORTHERN BOULEVARD`,
`NORTHERN BLVD`, and `Northern Blvd` were three rows, and so were
`GRAND CENTRAL PKWY` and `GRAND CENTRAL PARKWAY`. A ranking of streets tells a
reader where to look for dangerous roads, so a spelling should not move a
street down the list.

> **Once the spellings of each street are merged, which street has the most
> fatal crashes, and does the top ten change?**

### How to work

Work in pairs for 30 minutes. Submit a cleaned street column, a before-and-after
table of every value your rules changed, a new top ten, and a short
interpretation. If you run out of time, finish Task 6 after class.

- The row unit is **one fatal crash**, with the street name that the police
  recorded (`street`, built in lecture 10 from `on_street_name`).
- The setup reads `data/derived/traffic.duckdb`, which lecture 10's notebook
  builds. If the setup prints `False`, run lecture 10's notebook first.
- Each rule rewrites values. Print what it changed before you count anything.
- In the starter cells, replace each `...` with code.

In [1]:
import re
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option('display.max_rows', 30)

DERIVED = Path('../data/derived')
db_file = DERIVED / 'traffic.duckdb'
print(db_file.exists())   # False: run lecture 10's notebook first

con = duckdb.connect(str(db_file), read_only=True)
crashes = con.sql('SELECT collision_id, street FROM crashes').df()
con.close()

# Check: 3,469 crashes, one row each.
crashes.shape

True


(3469, 2)

## Task 1 · Rank the streets as recorded

Count the different street values and the crashes with no street. Then list
the ten values with the most crashes, as lecture 10 did. Finally, list every
spelling that contains `NORTHERN B`, capitals ignored, with its count.

How many crashes does Northern Boulevard have once its spellings are added
up, and where would that put it in the stored top ten?

In [2]:
crashes['street'].nunique(), crashes['street'].isna().sum()

(1186, 306)

In [3]:
crashes['street'].value_counts().head(10)

street
BELT PARKWAY               54
BROADWAY                   47
ATLANTIC AVENUE            43
3 AVENUE                   39
MAJOR DEEGAN EXPRESSWAY    35
GRAND CENTRAL PKWY         33
LINDEN BOULEVARD           30
HENRY HUDSON PARKWAY       28
BRUCKNER BOULEVARD         27
HYLAN BOULEVARD            27
Name: count, dtype: int64

In [4]:
northern = crashes['street'].str.contains('NORTHERN B', case=False,
                                          na=False)
crashes.loc[northern, 'street'].value_counts()

street
NORTHERN BOULEVARD    26
NORTHERN BLVD          3
Northern Blvd          1
Name: count, dtype: int64

The table holds **1,186** different street values, and **306** crashes have no
street. The Belt Parkway leads with 54 crashes, and the tenth value, Hylan
Boulevard, has 27. Northern Boulevard appears in three spellings, 26 + 3 + 1 =
**30** crashes, level with Linden Boulevard in seventh place. As stored, it is
not in the top ten.

## Task 2 · Standardize capitals and spaces

Make a column `street_clean`: the street in capitals, with every run of white
space replaced by one space, and no space at either end. How many different
values remain? Which street in the top ten gained a crash?

In [5]:
crashes['street_clean'] = (crashes['street'].str.upper()
                           .str.replace(r'\s+', ' ', regex=True)
                           .str.strip())
crashes['street_clean'].nunique()

1174

In [6]:
crashes['street_clean'].value_counts().head(10)

street_clean
BELT PARKWAY               54
BROADWAY                   47
ATLANTIC AVENUE            43
3 AVENUE                   39
MAJOR DEEGAN EXPRESSWAY    35
GRAND CENTRAL PKWY         34
LINDEN BOULEVARD           30
HENRY HUDSON PARKWAY       28
HYLAN BOULEVARD            27
LONG ISLAND EXPRESSWAY     27
Name: count, dtype: int64

**1,174** different values remain, 12 fewer. `Grand Central Pkwy` joined
`GRAND CENTRAL PKWY`, which rose from 33 to 34. Capitals and spaces explain
only a small part of the spellings; abbreviations are the larger part.

## Task 3 · Compare plain text with `\bST\b` in `.str.replace`

Replace `ST` with `STREET` in `street_clean` two ways: with `.str.replace`
as plain text, then with the pattern `\bST\b`. For each version, count the
different values that changed, and show a few of them before and after.

What goes wrong with plain text? Which value does `\bST\b` still get wrong?

In [7]:
before = crashes['street_clean']
plain = before.str.replace('ST', 'STREET', regex=False)
changed = before.notna() & (before != plain)
print(before[changed].nunique(), 'values changed')
pd.DataFrame({'before': before[changed],
              'after': plain[changed]}).drop_duplicates().head(6)

490 values changed


,before,after
0,BROAD STREET,BROAD STREETREET
2,WEST HOUSTON STREET,WESTREET HOUSTREETON STREETREET
3,CANAL STREET,CANAL STREETREET
6,ALLEN STREET,ALLEN STREETREET
7,EAST HOUSTON STREET,EASTREET HOUSTREETON STREETREET
8,WEST 41 STREET,WESTREET 41 STREETREET


In [8]:
bounded = before.str.replace(r'\bST\b', 'STREET', regex=True)
changed = before.notna() & (before != bounded)
print(before[changed].nunique(), 'values changed')
table = pd.DataFrame({'before': before[changed],
                      'after': bounded[changed]}).drop_duplicates()
table[table['before'].str.contains('^ST ')]

62 values changed


,before,after
702,ST JOHNS PLACE,STREET JOHNS PLACE


Plain text changes **490** values, almost all of them wrongly: `ST` inside a
word becomes `STREET`, so `WEST HOUSTON STREET` turns into
`WESTREET HOUSTREETON STREETREET`. With `\bST\b`, `ST` must be a whole word,
and **62** values change, such as `W 86 ST`. One is still wrong:
`ST JOHNS PLACE` becomes `STREET JOHNS PLACE`, because `ST` at the start of a
name means Saint.

## Task 4 · Expand the abbreviations with a dictionary of rules

Complete the dictionary `rules`, which maps a pattern to the full word. The
loop below it applies every rule in order to `street_clean`, as the lecture's
loop labeled each vehicle, and stores the result as `street_full`. Then:

1. Add the rule that turns `ST` at the **start** of a name into `SAINT`, before
   the `STREET` rule.
2. Add rules that turn `E`, `W`, `N`, and `S` into `EAST`, `WEST`, `NORTH`,
   and `SOUTH`, **only at the start** of a name.
3. Count the crashes that changed and the different values that remain.
   Then print, before and after, every changed value that your new rules
   touched: `ST` or a compass letter at the start of the name.
4. List the values of `street_clean` in which `S` is a whole word, with
   their `street_full`. Which of them changed?

Why must the compass rules use `^`? Look at `AVENUE S` and
`THOMAS S BOYLAND STREET` in the last table.

In [9]:
rules = {
    r'^ST\b': 'SAINT',
    r'\bST\b': 'STREET',
    r'\bAVE\b': 'AVENUE',
    r'\bBLVD\b': 'BOULEVARD',
    r'\bPKWY\b': 'PARKWAY',
    r'\bEXPY\b|\bEXPWY\b': 'EXPRESSWAY',
    r'\bRD\b': 'ROAD',
    r'\bDR\b': 'DRIVE',
    r'\bPL\b': 'PLACE',
    r'^E\b': 'EAST', r'^W\b': 'WEST',
    r'^N\b': 'NORTH', r'^S\b': 'SOUTH',
}

street_full = crashes['street_clean']
for pattern, full in rules.items():
    street_full = street_full.str.replace(pattern, full, regex=True)
crashes['street_full'] = street_full

In [10]:
changed = (crashes['street_clean'].notna()
           & (crashes['street_clean'] != crashes['street_full']))
print(changed.sum(), 'crashes changed;',
      crashes['street_full'].nunique(), 'different values remain')
new_rules = crashes['street_clean'].str.contains(r'^ST\b|^[NSEW]\b',
                                                 na=False)
(crashes.loc[changed & new_rules, ['street_clean', 'street_full']]
 .drop_duplicates().sort_values('street_clean'))

324 crashes changed; 1044 different values remain


,street_clean,street_full
3365,E 118 ST,EAST 118 STREET
3167,E 12 ST,EAST 12 STREET
3294,E 14 ST,EAST 14 STREET
3230,E 149 ST,EAST 149 STREET
3386,E 222 ST,EAST 222 STREET
3295,E 232 ST,EAST 232 STREET
3345,E 40 ST,EAST 40 STREET
3250,E 53 ST,EAST 53 STREET
3277,E 8 ST,EAST 8 STREET
3309,E 83 ST,EAST 83 STREET


In [11]:
single_s = crashes['street_clean'].str.contains(r'\bS\b', na=False)
(crashes.loc[single_s, ['street_clean', 'street_full']]
 .drop_duplicates())

,street_clean,street_full
136,AVENUE S,AVENUE S
2924,THOMAS S BOYLAND STREET,THOMAS S BOYLAND STREET
3319,S CONDUIT AVE,SOUTH CONDUIT AVENUE
3449,WEST MOSHOLU PKWY S,WEST MOSHOLU PARKWAY S
3461,S GANNON AVE,SOUTH GANNON AVENUE


The rules rewrote the street of **324** crashes, and **1,044** different
values remain. The new rules touched 29 different values: `ST JOHNS PLACE`
becomes `SAINT JOHNS PLACE`, `E 149 ST` becomes `EAST 149 STREET`,
`N CONDUIT AVE` becomes `NORTH CONDUIT AVENUE`, and `W 86 ST` becomes
`WEST 86 STREET`. Each `after` reads as a full street name.

Five values hold `S` as a whole word. Only `S CONDUIT AVE` and
`S GANNON AVE` changed. `AVENUE S` is one of Brooklyn's lettered avenues,
where `S` is a name, not South; in `THOMAS S BOYLAND STREET` it is a middle
initial; `WEST MOSHOLU PKWY S` names the south side of a parkway. A rule
`\bS\b` without `^` would rewrite all three.

## Task 5 · Rank the cleaned streets and decide two merges

1. List the twelve values of `street_full` with the most crashes. Which street
   entered the top ten, and which left it?
2. Some values name a ramp, an exit, or a service road beside a highway.
   Count the crashes whose `street_full` contains one of the words `RAMP`,
   `SERVICE`, `SVC`, `EXIT`, or `ENTRANCE`, and list them.
3. Third Avenue still has three spellings. List the values that contain
   `3 AVENUE`, `3RD AVENUE`, or `THIRD AVENUE`. Why is
   `.str.contains('3 AVENUE')` the wrong test?

Would you merge the ramps with their highway, and the spellings of Third
Avenue? Say how each choice changes the top of the list.

In [12]:
crashes['street_full'].value_counts().head(12)

street_full
BELT PARKWAY               55
BROADWAY                   47
ATLANTIC AVENUE            44
3 AVENUE                   43
GRAND CENTRAL PARKWAY      37
MAJOR DEEGAN EXPRESSWAY    37
LINDEN BOULEVARD           32
BRUCKNER BOULEVARD         32
NORTHERN BOULEVARD         30
HYLAN BOULEVARD            30
HENRY HUDSON PARKWAY       28
LONG ISLAND EXPRESSWAY     27
Name: count, dtype: int64

In [13]:
near_highway = crashes['street_full'].str.contains(
    r'\b(?:RAMP|SERVICE|SVC|EXIT|ENTRANCE)\b', na=False)
print(near_highway.sum(), 'crashes')
crashes.loc[near_highway, 'street_full'].value_counts().head(8)

27 crashes


street_full
GOWANUS RAMP                           3
MAJOR DEEGAN EXPRESSWAY RAMP           3
HARLEM RIVER DRIVE RAMP                2
TUNNEL EXIT STREET                     2
BROOKLYN QUEENS EXPRESSWAY RAMP        2
CLEARVIEW EXPRESSWAY RAMP              1
RFK BRIDGE EXIT 125 STREET 2 AVENUE    1
BELT PARKWAY RAMP                      1
Name: count, dtype: int64

In [14]:
third = crashes['street_full'].str.contains(
    r'^(?:3|3RD|THIRD) AVENUE$', na=False)
print(crashes.loc[third, 'street_full'].value_counts())
crashes['street_full'].str.contains('3 AVENUE', na=False).sum()

street_full
3 AVENUE        43
THIRD AVENUE     2
3RD AVENUE       1
Name: count, dtype: int64


60

The Belt Parkway still leads, with **55** crashes. **Northern Boulevard** (30)
entered the top ten, and the **Henry Hudson Parkway** (28) left it. The Grand
Central Parkway rose from 33 to 37 crashes, now level with the Major Deegan
Expressway.

**27** crashes name a ramp, an exit, or a service road. They are places next
to a highway, not on it, so a ranking of roads can keep them apart; merging
them would add three crashes to the Major Deegan Expressway. Third Avenue has
43 + 2 + 1 = 46 crashes across three spellings, which would move it above
Atlantic Avenue (44) to third place. `.str.contains('3 AVENUE')` also matches
`73 AVENUE`, `13 AVENUE`, and other avenues: it counts 60 crashes, and it
misses `3RD AVENUE` and `THIRD AVENUE`.

## Task 6 · Interpret the cleaned ranking

Write three or four sentences. Which street has the most fatal crashes after
your rules, and how sure are you of the top three? Name two choices your rules
made, and say how each one moves a street in the ranking.

After the rules, the Belt Parkway has the most fatal crashes (55), ahead of
Broadway (47). Third place depends on a choice: Atlantic Avenue (44) as the
rules stand, or Third Avenue (46) if its three spellings are merged. The rules
read `ST` at the start of a name as Saint and kept ramps and service roads
apart from their highways; merging the ramps would add three crashes to the
Major Deegan Expressway. The 306 crashes with no street are missing from every
count.

## Checks for a rule that rewrites values

- A rule that rewrites values can create new errors. Print every value it
  changed, before and after, and list the real values that look like a match
  but should not change.
- `\b` keeps a rule inside whole words; `^` and `$` keep it at the start or the
  end. Brooklyn's `AVENUE S` shows why both matter.
- Each merge is a decision that can reorder a ranking. Report the decisions
  next to the ranking.